# Notebook 1 — Download Amazon Reviews 2023 `Video_Games` raw data

This notebook is the first step of the project pipeline. It is designed to:

- download the **Amazon Reviews 2023** data for the **`Video_Games`** category only
- save the raw files into the repository's `data/raw/` folder
- keep the raw data as close to the source as possible
- do a light validation step with Spark so the next notebook can start from local files

## Output of this notebook

The notebook will save:

- `reviews.jsonl`
- `metadata.jsonl`


## 1. Optional package installation

Run this cell only if your environment is missing the required packages or if the
installed `datasets` version is too new for this dataset.

In [ ]:
# Uncomment if needed
# %pip install datasets==2.17.0 pyarrow pyspark

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/536.6 kB ? eta -:--:--
   ---------------------------------------  524.3/536.6 kB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 536.6/536.6 kB 2.5 MB/s  0:00:00

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2026.2.0

    Uninstalling fsspec-2026.2.0:

      Successfully uninstalled fsspec-2026.2.0

   -------- ------------------------------- 1/5 [fsspec]
   -------- ------------------------------- 1/5 [fsspec]
   -------- ------------------------------- 1/5 [fsspec]
  Attempting uninstall: dill
   -------- ------------------------------- 1/5 [fsspec]
    Found existing installation: dill 0.4.1
   -------- ------------------------------- 1/5 [fsspec]
    Uninstalling dill-0.4.1:
   -------- ------------------------------- 1/5 [fsspec]
      Success

## 2. Imports and output paths

This cell imports the required libraries and defines the file paths where the raw review data,
metadata, and download manifest will be stored.

In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone

from datasets import load_dataset
from pyspark.sql import SparkSession

raw_dir = Path("../data/raw/amazon_reviews_2023/video_games")
raw_dir.mkdir(parents=True, exist_ok=True)

reviews_path = raw_dir / "reviews.jsonl"
metadata_path = raw_dir / "metadata.jsonl"
manifest_path = raw_dir / "download_manifest.json"

print(f"Raw output folder: {raw_dir.resolve()}")

c:\Users\olive\anaconda3\envs\pyspark_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw output folder: C:\Users\olive\git\HAMK\BigDataAnalytics\Project\big-data-video-games-reviews\data\raw\amazon_reviews_2023\video_games


## 3. Define the dataset configuration

According to the dataset card, the naming pattern is:

- `raw_review_<CATEGORY>`
- `raw_meta_<CATEGORY>`

For this project we only want the `Video_Games` category.


In [2]:
DATASET_NAME = "McAuley-Lab/Amazon-Reviews-2023"
CATEGORY = "Video_Games"

REVIEW_CONFIG = f"raw_review_{CATEGORY}"
META_CONFIG = f"raw_meta_{CATEGORY}"

print("Dataset:", DATASET_NAME)
print("Review config:", REVIEW_CONFIG)
print("Metadata config:", META_CONFIG)

Dataset: McAuley-Lab/Amazon-Reviews-2023
Review config: raw_review_Video_Games
Metadata config: raw_meta_Video_Games


## 4. Download helper function

The `Video_Games` subset is large, so the data is streamed and written line by line
to a local `.jsonl` file instead of loading everything into memory at once.

Set `OVERWRITE = True` if you want to download the files again.

In [3]:
OVERWRITE = False
PROGRESS_EVERY = 100_000

def stream_to_jsonl(dataset_name, config_name, output_path, overwrite=False):
    if output_path.exists() and not overwrite:
        raise FileExistsError(
            f"{output_path} already exists. Set OVERWRITE = True to download again."
        )

    dataset = load_dataset(
        dataset_name,
        config_name,
        split="full",
        streaming=True,
        trust_remote_code=True,
    )

    row_count = 0
    with output_path.open("w", encoding="utf-8") as f:
        for row_count, record in enumerate(dataset, start=1):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            if row_count % PROGRESS_EVERY == 0:
                print(f"{config_name}: wrote {row_count:,} rows...")

    print(f"{config_name}: finished with {row_count:,} rows")
    return row_count

def file_size_mb(path):
    return round(path.stat().st_size / (1024 ** 2), 2)

## 5. Download raw review data

This writes one JSON object per line to `reviews.jsonl`.


In [4]:
review_rows = stream_to_jsonl(
    DATASET_NAME,
    REVIEW_CONFIG,
    reviews_path,
    overwrite=OVERWRITE,
)

print(f"Review file size: {file_size_mb(reviews_path):,.2f} MB")

raw_review_Video_Games: wrote 100,000 rows...
raw_review_Video_Games: wrote 200,000 rows...
raw_review_Video_Games: wrote 300,000 rows...
raw_review_Video_Games: wrote 400,000 rows...
raw_review_Video_Games: wrote 500,000 rows...
raw_review_Video_Games: wrote 600,000 rows...
raw_review_Video_Games: wrote 700,000 rows...
raw_review_Video_Games: wrote 800,000 rows...
raw_review_Video_Games: wrote 900,000 rows...
raw_review_Video_Games: wrote 1,000,000 rows...
raw_review_Video_Games: wrote 1,100,000 rows...
raw_review_Video_Games: wrote 1,200,000 rows...
raw_review_Video_Games: wrote 1,300,000 rows...
raw_review_Video_Games: wrote 1,400,000 rows...
raw_review_Video_Games: wrote 1,500,000 rows...
raw_review_Video_Games: wrote 1,600,000 rows...
raw_review_Video_Games: wrote 1,700,000 rows...
raw_review_Video_Games: wrote 1,800,000 rows...
raw_review_Video_Games: wrote 1,900,000 rows...
raw_review_Video_Games: wrote 2,000,000 rows...
raw_review_Video_Games: wrote 2,100,000 rows...
raw_review

## 6. Download raw metadata

This writes one JSON object per line to `metadata.jsonl`.


In [5]:
metadata_rows = stream_to_jsonl(
    DATASET_NAME,
    META_CONFIG,
    metadata_path,
    overwrite=OVERWRITE,
)

print(f"Metadata file size: {file_size_mb(metadata_path):,.2f} MB")

raw_meta_Video_Games: wrote 100,000 rows...
raw_meta_Video_Games: finished with 137,269 rows
Metadata file size: 405.62 MB


## 7. Validate the downloaded files with Spark

This cell loads the local raw files into Spark and prints a quick schema/sample preview.

This is a validation step only.


In [6]:
spark = SparkSession.builder.appName("Notebook1_Download_Amazon_Video_Games").getOrCreate()

reviews_df = spark.read.json(str(reviews_path))
metadata_df = spark.read.json(str(metadata_path))

print("Spark loaded the local JSON files successfully.")

Spark loaded the local JSON files successfully.


In [7]:
print("=== Reviews schema ===")
reviews_df.printSchema()

print("=== Reviews sample ===")
reviews_df.show(5, truncate=False)

=== Reviews schema ===
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

=== Reviews sample ===
+----------+------------+------+-----------+------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
print("=== Metadata schema ===")
metadata_df.printSchema()

print("=== Metadata sample ===")
metadata_df.show(5, truncate=False)

=== Metadata schema ===
root
 |-- author: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- details: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- images: struct (nullable = true)
 |    |-- hi_res: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- large: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- thumb: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- variant: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- price: string (nullable =